# 04 — Deep Fetch: Language Enrichment + Tier-B Subsample  
### *Coding Alone* · ICSE 2027 SEIS pipeline

**Self-contained.** Uses the **GitHub GraphQL API** (not BigQuery) to:
1. **Fix the 94%-unknown language gap** — fetch `primaryLanguage` for all 50k frame repos (needed for the AI-exposure difference-in-differences). Cheap: ~1,000 batched queries.
2. Select the **Tier-B subsample** (~2,000 stratified repos) whose comment *text* we'll fetch in **04b** for the Mentorship Decay Index.

**Interruption-safe:** batches checkpoint to Drive JSONL + a manifest; GitHub rate limits trigger automatic sleep-until-reset; re-running skips finished batches. No data lost on disconnect or rate-limit timeout.

**You need a GitHub token** (free). Steps in the token cell below.


## Config — edit PROJECT_ID

In [ ]:
# ==== EDIT ME ==============================================================
PROJECT_ID = "YOUR-GCP-PROJECT-ID"
# ==========================================================================
BQ_DATASET = "seis"; BQ_LOCATION = "US"
DRIVE_BASE = "/content/drive/MyDrive/ICSE_SEIS"
TIER_B_N = 2000            # size of the deep-fetch subsample for MDI/annotation
assert PROJECT_ID != "PASTE-YOUR-PROJECT-ID-HERE", "Set PROJECT_ID first."
print("Config loaded.")


## Setup — auth, BigQuery, Drive, helpers

In [ ]:
import os, json, time, hashlib
import pandas as pd, numpy as np
import requests
from google.colab import auth, drive
from google.cloud import bigquery

auth.authenticate_user()
client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
drive.mount('/content/drive')
for sub in ["", "/data/derived", "/checkpoints", "/tables"]:
    os.makedirs(DRIVE_BASE + sub, exist_ok=True)
CKPT_DIR = f"{DRIVE_BASE}/checkpoints"
def _atomic(p,t):
    tmp=p+".tmp"; open(tmp,"w").write(t); os.replace(tmp,p)
def load_json(n,d):
    p=f"{CKPT_DIR}/{n}"; return json.load(open(p)) if os.path.exists(p) else d
def save_json(n,o): _atomic(f"{CKPT_DIR}/{n}", json.dumps(o,default=str))
P = f"{PROJECT_ID}.{BQ_DATASET}"
print("Ready. Dataset:", P)


## GitHub token — one-time setup
1. Go to **github.com → Settings → Developer settings → Personal access tokens → Tokens (classic) → Generate new token (classic)**. No scopes needed for public data (or tick `public_repo`). Copy the token.
2. In Colab's left sidebar click the **🔑 key icon (Secrets)** → **Add new secret** → name it exactly `GITHUB_TOKEN`, paste the token as the value, and enable **Notebook access**.
3. Run the cell below. (Storing it as a Colab Secret keeps the token out of the notebook so it's safe to share.)

In [ ]:
from google.colab import userdata
try:
    TOKEN = userdata.get('GITHUB_TOKEN')
    assert TOKEN
except Exception:
    raise RuntimeError("Add a Colab Secret named GITHUB_TOKEN (key icon in left sidebar), enable notebook access, then re-run.")
GQL = "https://api.github.com/graphql"
HEAD = {"Authorization": f"bearer {TOKEN}"}
# sanity check
who = requests.post(GQL, json={"query":"{ viewer { login } rateLimit { remaining } }"}, headers=HEAD).json()
print("Token OK. GraphQL rate remaining:", who.get("data",{}).get("rateLimit",{}).get("remaining"))


## Rate-limited GraphQL client
Sleeps until reset when the budget runs low, honors secondary-limit `Retry-After`, and retries transient 5xx with backoff — so a rate-limit or network blip pauses and resumes rather than failing.

In [ ]:
def gql(query, max_tries=6):
    for i in range(max_tries):
        try:
            r = requests.post(GQL, json={"query": query}, headers=HEAD, timeout=60)
            if r.status_code == 200:
                j = r.json()
                rl = (j.get("data") or {}).get("rateLimit")
                if rl and rl.get("remaining", 999) < 50:
                    reset = pd.Timestamp(rl["resetAt"]); now = pd.Timestamp.utcnow().tz_localize("UTC")
                    wait = max(0, (reset - now).total_seconds()) + 5
                    print(f"  rate low ({rl['remaining']}); sleeping {wait:.0f}s"); time.sleep(wait)
                return j
            if r.status_code in (403, 429):
                ra = int(r.headers.get("Retry-After", "60")); print(f"  {r.status_code}; sleep {ra}s"); time.sleep(ra); continue
            if r.status_code >= 500:
                time.sleep(2**i); continue
            print("  HTTP", r.status_code, r.text[:200]); time.sleep(2**i)
        except Exception as e:
            print("  req error:", e); time.sleep(2**i)
    raise RuntimeError("gql failed after retries")
print("gql() ready.")


## 1. Language enrichment — all 50k repos (batched, resumable)
Pulls `primaryLanguage`, current stars, fork/archive flags, and creation date for every frame repo, 50 per GraphQL call. Each batch appends to a Drive JSONL and is checkpointed; re-running resumes. Deleted/renamed repos return null → language stays unknown (recorded, not an error).

In [ ]:
names = client.query(f"SELECT repo_name FROM `{P}.sampling_frame`").result().to_dataframe()["repo_name"].tolist()
pairs = [tuple(n.split('/',1)) for n in names if isinstance(n,str) and '/' in n]
BATCH = 50
batches = [pairs[i:i+BATCH] for i in range(0, len(pairs), BATCH)]
jsonl = f"{DRIVE_BASE}/data/derived/repo_language.jsonl"
done = set(load_json("manifest_lang_batches.json", []))
print(f"{len(pairs):,} repos in {len(batches):,} batches; {len(done):,} batches already done.")

def build_q(batch):
    parts = [f'r{i}: repository(owner:{json.dumps(o)}, name:{json.dumps(n)}) '
             f'{{ nameWithOwner primaryLanguage{{name}} stargazerCount isFork isArchived createdAt }}'
             for i,(o,n) in enumerate(batch)]
    return "query{\n" + "\n".join(parts) + "\nrateLimit{remaining resetAt}}"

for bi, batch in enumerate(batches):
    if str(bi) in done: continue
    data = (gql(build_q(batch)).get("data") or {})
    rows = []
    for i,(o,n) in enumerate(batch):
        r = data.get(f"r{i}") or {}
        lang = (r.get("primaryLanguage") or {}).get("name")
        rows.append({"repo_name": f"{o}/{n}", "primary_language_api": lang,
                     "stars_api": r.get("stargazerCount"), "is_fork": r.get("isFork"),
                     "is_archived": r.get("isArchived"), "created_at_api": r.get("createdAt")})
    with open(jsonl, "a") as f:
        for row in rows: f.write(json.dumps(row) + "\n")
    done.add(str(bi)); save_json("manifest_lang_batches.json", sorted(done, key=int))
    if bi % 25 == 0: print(f"  batch {bi}/{len(batches)}")
print("Language fetch complete.")


## 2. Load languages to BigQuery + build enriched frame

In [ ]:
lang_df = pd.read_json(jsonl, lines=True).drop_duplicates("repo_name")
client.load_table_from_dataframe(
    lang_df, f"{P}.repo_language",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")).result()
print("repo_language rows:", len(lang_df))

enrich_sql = f"""
SELECT f.*,
  COALESCE(NULLIF(l.primary_language_api,''), f.primary_language, 'unknown') AS language,
  l.stars_api, l.is_fork, l.is_archived, l.created_at_api
FROM `{P}.sampling_frame` f
LEFT JOIN `{P}.repo_language` l USING (repo_name)
"""
client.query(enrich_sql, job_config=bigquery.QueryJobConfig(
    destination=f"{P}.sampling_frame_enriched", write_disposition="WRITE_TRUNCATE")).result()
cov = client.query(f"""
SELECT COUNTIF(language!='unknown')/COUNT(*) AS lang_coverage, COUNT(*) n
FROM `{P}.sampling_frame_enriched`""").result().to_dataframe()
print("Language coverage now:", round(float(cov.lang_coverage.iloc[0])*100,1), "%")
display(client.query(f"""
SELECT language, COUNT(*) n FROM `{P}.sampling_frame_enriched`
GROUP BY language ORDER BY n DESC LIMIT 15""").result().to_dataframe())


## 3. Tier-B subsample (~2,000 repos, stratified)
Stratified draw from the enriched frame (by stars × contributors × activity × language) — the set whose comment text 04b will fetch for the Mentorship Decay Index and the 500-interaction manual annotation. Prefer repos with real PR/issue discussion so there's text to analyze.

In [ ]:
fr = client.query(f"""
SELECT repo_id, repo_name, total_stars, peak_year_actors, total_events, age_days,
       total_pr, total_issues, language
FROM `{P}.sampling_frame_enriched`
WHERE (total_pr + total_issues) >= 5           -- ensure some discussion to analyze
""").result().to_dataframe()

def qbin(s,k=4):
    try: return pd.qcut(s.rank(method="first"), k, labels=False)
    except Exception: return pd.cut(s, k, labels=False)
fr["sb"]=qbin(fr.total_stars); fr["cb"]=qbin(fr.peak_year_actors); fr["ab"]=qbin(fr.total_events)
top = fr.language.value_counts().head(10).index
fr["lb"] = np.where(fr.language.isin(top), fr.language, "other")
g = fr.groupby(["sb","cb","ab","lb"], observed=True)
frac = min(1.0, TIER_B_N/len(fr))
tb = g.apply(lambda x: x.sample(n=min(len(x), max(1, round(len(x)*frac))), random_state=42)).reset_index(drop=True)
if len(tb) > TIER_B_N: tb = tb.sample(n=TIER_B_N, random_state=7)
client.load_table_from_dataframe(
    tb[["repo_id","repo_name","total_stars","peak_year_actors","total_events","age_days","language"]],
    f"{P}.tier_b_frame", job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")).result()
print(f"Tier-B subsample: {len(tb)} repos -> {P}.tier_b_frame")
tb.language.value_counts().head(10)


## Done ✅
Fixed the language gap (`sampling_frame_enriched` with a real `language` column) and selected the Tier-B subsample (`tier_b_frame`).

Next: **`04b_deep_fetch_text.ipynb`** — fetch issue/PR **comment text** and newcomer first-contribution timing for the Tier-B repos (resumable, rate-limited), which feeds the **Mentorship Decay Index (MDI)** and the 500-interaction manual annotation set.